In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 2: Ads CTR Prediction & Ranking

### Business Context
Sponsored search auction. Rank ads using a Generalized Second Price (GSP) auction.  
**Rank score** = bid × pCTR × quality_score. Winner pays the second-highest bid.

**Critical**: pCTR must be **calibrated** (not just a ranking score) because it directly affects the price each advertiser pays. Miscalibration → advertiser churn or lost revenue.

**Scale**: 100k queries/sec, <10ms SLA per query.

### Key Concepts

**1. Position Bias in Ads:**
Ads shown at position 1 get 3-5x more clicks than position 3, regardless of quality.  
Training on raw clicks without correction inflates pCTR for top positions.

**2. Inverse Propensity Scoring (IPS):**
Weight each training example by `1 / P(shown at position k)`.  
Corrects for the fact that high positions are seen more. Unbiased estimator of true CTR.

**3. Calibration:**
Model outputs `P(click) = 0.05` should match empirical click rate of 5%.  
Check with reliability diagram (calibration curve). Fix with Platt scaling (logistic on top of scores) or isotonic regression.

**4. Two-Stage Architecture:**
- Stage 1 (Retrieval): FAISS ANN over ad embeddings, ~500 candidates
- Stage 2 (Ranking): GBDT or deep model on full feature set, <10ms

In [ ]:
# --- IPS-weighted CTR model with Platt scaling calibration ---

def generate_ads_data(n=5000):
    """Synthetic ads click data with position bias."""
    positions = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    # Propensity: probability of being shown at each position
    propensity_map = {1: 0.30, 2: 0.25, 3: 0.20, 4: 0.15, 5: 0.10}
    propensities = np.array([propensity_map[p] for p in positions])

    # True CTR (position-independent relevance)
    ad_quality = np.random.beta(2, 5, size=n)      # true ad relevance
    position_multiplier = np.array([3.0, 2.0, 1.5, 1.2, 1.0])[positions - 1]
    observed_ctr = ad_quality * position_multiplier
    clicks = (np.random.rand(n) < observed_ctr.clip(0, 1)).astype(int)

    features = np.column_stack([
        ad_quality + np.random.randn(n) * 0.1,   # noisy quality signal
        np.random.randn(n, 4)                     # other features
    ])
    return features, clicks, positions, propensities

X, y, positions, propensities = generate_ads_data()
X_train, X_test, y_train, y_test, pos_train, pos_test, prop_train, prop_test = \
    train_test_split(X, y, positions, propensities, test_size=0.2, random_state=42)

# --- Model 1: Naive (no IPS) ---
naive_model = LogisticRegression(max_iter=1000)
naive_model.fit(X_train, y_train)
naive_preds = naive_model.predict_proba(X_test)[:, 1]

# --- Model 2: IPS-weighted ---
ips_weights = 1.0 / prop_train
ips_weights = np.clip(ips_weights, 1, 20)   # clip to avoid high variance
ips_model = LogisticRegression(max_iter=1000)
ips_model.fit(X_train, y_train, sample_weight=ips_weights)
ips_preds = ips_model.predict_proba(X_test)[:, 1]

# --- Model 3: IPS + Platt scaling calibration ---
calibrated_model = CalibratedClassifierCV(
    LogisticRegression(max_iter=1000), method='sigmoid', cv=3
)
calibrated_model.fit(X_train, y_train, sample_weight=ips_weights)
cal_preds = calibrated_model.predict_proba(X_test)[:, 1]

print('Model Comparison (Test Set):')
print(f'  Naive        AUC: {roc_auc_score(y_test, naive_preds):.4f}  LogLoss: {log_loss(y_test, naive_preds):.4f}')
print(f'  IPS          AUC: {roc_auc_score(y_test, ips_preds):.4f}  LogLoss: {log_loss(y_test, ips_preds):.4f}')
print(f'  IPS+Calibrat AUC: {roc_auc_score(y_test, cal_preds):.4f}  LogLoss: {log_loss(y_test, cal_preds):.4f}')

# --- Expected Calibration Error (ECE) ---
def expected_calibration_error(y_true, y_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_conf = y_pred[mask].mean()
        bin_acc  = y_true[mask].mean()
        ece += mask.sum() / len(y_true) * abs(bin_conf - bin_acc)
    return ece

print(f'\nECE (lower is better):')
print(f'  Naive:       {expected_calibration_error(y_test, naive_preds):.4f}')
print(f'  IPS+Calibrat:{expected_calibration_error(y_test, cal_preds):.4f}')

### L6 Interview Q&A

**Q: Why does miscalibration hurt revenue in a GSP auction?**  
A: In GSP, the price each winner pays depends on the competitor's bid × pCTR. If pCTR is systematically overestimated, winners overpay → advertiser churn. If underestimated, the platform undercharges → lost revenue. Calibration ensures pCTR equals empirical click probability across score buckets.

**Q: What's the difference between IPS and including position as a feature?**  
A: IPS reweights the loss function to correct for position bias in training. Position-as-feature includes position at training time so the model learns position effects, then zeros it out at inference to get position-independent scores. IPS is unbiased but high variance (clipping helps). Position-as-feature is simpler and often works better in practice. Both are valid; many systems use both.

**Q: Your model has AUC=0.82 but ECE=0.08. Is it production-ready?**  
A: High AUC means good ranking but ECE=0.08 means poor calibration — the model is systematically over/underconfident. For feed ranking alone, it might be fine. For ads, it's not — we need calibrated probabilities to compute fair auction prices. Apply Platt scaling or isotonic regression to fix ECE before deployment.

### Common Pitfalls
- Using ranking AUC as the sole metric when calibration matters for the business logic
- Not clipping IPS weights — extreme weights cause high variance and training instability
- Forgetting to zero out position feature at inference time